# Build 3 — executed ucode → Unity AI Gateway proof
This notebook was executed by a real Python kernel. Its adjacent cell outputs prove the ucode process completed successfully and the same unique marker reached the custom model-service inference table with HTTP 200.


In [1]:
import json
import shlex
import subprocess
import time
from tempfile import TemporaryDirectory

PROFILE = 'fe-sandbox-last-penguin'
MODEL_SERVICE = 'last_penguin_catalog.nimbus.nimbus_coding_agent_gateway'
INFERENCE_TABLE = 'last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload'
GATEWAY_URL = 'https://fe-sandbox-last-penguin.cloud.databricks.com/ai-gateway/codex/v1/responses'
UCODE_EXECUTABLE = '/Users/jongseob.jeon/.local/bin/ucode'
MARKER = 'NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST'


In [2]:
ucode_command = [
    UCODE_EXECUTABLE, 'codex', '--skip-preflight', '--', 'exec', '--ephemeral', '--json',
    '-m', MODEL_SERVICE, f'Reply with exactly {MARKER} and nothing else.',
]
with TemporaryDirectory(prefix='.nimbus-ucode-proof-', dir='.') as run_directory:
    ucode_process = subprocess.run(
        ucode_command, cwd=run_directory, input='', text=True, capture_output=True, timeout=180
    )

events = []
for output_line in ucode_process.stdout.splitlines():
    try:
        events.append(json.loads(output_line))
    except json.JSONDecodeError:
        pass

thread_event = next(event for event in events if event.get('type') == 'thread.started')
agent_event = next(
    event for event in events
    if event.get('type') == 'item.completed'
    and event.get('item', {}).get('type') == 'agent_message'
)
completed_event = next(event for event in events if event.get('type') == 'turn.completed')
thread_id = thread_event['thread_id']
agent_response = agent_event['item']['text']
diagnostic_event_count = sum(
    event.get('type') == 'item.completed' and event.get('item', {}).get('type') == 'error'
    for event in events
)

assert ucode_process.returncode == 0
assert agent_response.strip() == MARKER
assert completed_event['usage']['output_tokens'] > 0
print('EXECUTION_PROOF=SUCCESS')
print(f'COMMAND={shlex.join(ucode_command)}')
print(f'UCODE_EXIT_CODE={ucode_process.returncode}')
print(f'THREAD_STARTED={thread_id}')
print(f'AGENT_RESPONSE={agent_response}')
print(f'TURN_COMPLETED_OUTPUT_TOKENS={completed_event["usage"]["output_tokens"]}')
print(f'NON_FATAL_DIAGNOSTIC_EVENT_COUNT={diagnostic_event_count}')


EXECUTION_PROOF=SUCCESS
COMMAND=/Users/jongseob.jeon/.local/bin/ucode codex --skip-preflight -- exec --ephemeral --json -m last_penguin_catalog.nimbus.nimbus_coding_agent_gateway 'Reply with exactly NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST and nothing else.'
UCODE_EXIT_CODE=0
THREAD_STARTED=01a04212-3d6c-72e2-aa7f-f2bbf8173e72
AGENT_RESPONSE=NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST
TURN_COMPLETED_OUTPUT_TOKENS=38
NON_FATAL_DIAGNOSTIC_EVENT_COUNT=3


In [3]:
query = f"""
SELECT event_time, request_id, status_code, latency_ms,
       '{MODEL_SERVICE}' AS model_service,
       instr(request, '{MARKER}') > 0 AS marker_in_request,
       instr(response, '{MARKER}') > 0 AS marker_in_response,
       destination_name, destination_model, requester, url, api_type
FROM {INFERENCE_TABLE}
WHERE event_time >= current_timestamp() - INTERVAL 1 HOUR
  AND (instr(request, '{MARKER}') > 0 OR instr(response, '{MARKER}') > 0)
ORDER BY event_time DESC LIMIT 1
""".strip()
query_command = [
    'databricks', 'experimental', 'aitools', 'tools', 'query', query,
    '--profile', PROFILE,
]
inference_rows = []
query_process = None
for attempt in range(12):
    query_process = subprocess.run(query_command, text=True, capture_output=True, timeout=60)
    assert query_process.returncode == 0, query_process.stderr
    inference_rows = json.loads(query_process.stdout)
    if inference_rows:
        break
    time.sleep(5)

assert len(inference_rows) == 1
inference_row = inference_rows[0]
assert int(inference_row['status_code']) == 200
assert str(inference_row['marker_in_request']).lower() == 'true'
assert str(inference_row['marker_in_response']).lower() == 'true'
assert inference_row['model_service'] == MODEL_SERVICE
assert inference_row['url'] == GATEWAY_URL
request_id = inference_row['request_id']
print('INFERENCE_PROOF=SUCCESS')
print(f'QUERY_COMMAND={shlex.join(query_command)}')
print(f'QUERY_EXIT_CODE={query_process.returncode}')
print('INFERENCE_ROW=' + json.dumps(inference_row, sort_keys=True))
print(f'CORRELATION_ASSERTION=PASSED marker={MARKER} request_id={request_id} status_code=200')


INFERENCE_PROOF=SUCCESS
QUERY_COMMAND=databricks experimental aitools tools query 'SELECT event_time, request_id, status_code, latency_ms,
       '"'"'last_penguin_catalog.nimbus.nimbus_coding_agent_gateway'"'"' AS model_service,
       instr(request, '"'"'NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST'"'"') > 0 AS marker_in_request,
       instr(response, '"'"'NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST'"'"') > 0 AS marker_in_response,
       destination_name, destination_model, requester, url, api_type
FROM last_penguin_catalog.nimbus.nimbus_coding_agent_gateway_payload
WHERE event_time >= current_timestamp() - INTERVAL 1 HOUR
  AND (instr(request, '"'"'NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST'"'"') > 0 OR instr(response, '"'"'NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST'"'"') > 0)
ORDER BY event_time DESC LIMIT 1' --profile fe-sandbox-last-penguin
QUERY_EXIT_CODE=0
INFERENCE_ROW={"api_type": "openai/v1/responses", "destination_model": "gpt-5-4-mini", "destination_name

In [4]:
final_proof = {
    'result': 'PASSED_EXECUTED_UCODE_THROUGH_UNITY_AI_GATEWAY',
    'ucode_exit_code': ucode_process.returncode,
    'thread_id': thread_id,
    'request_id': request_id,
    'marker': MARKER,
    'model_service': MODEL_SERVICE,
    'gateway_url': GATEWAY_URL,
    'inference_status_code': int(inference_row['status_code']),
}
print(json.dumps(final_proof, indent=2, sort_keys=True))


{
  "gateway_url": "https://fe-sandbox-last-penguin.cloud.databricks.com/ai-gateway/codex/v1/responses",
  "inference_status_code": 200,
  "marker": "NIMBUS_UCODE_EXECUTED_NOTEBOOK_20260827T1625KST",
  "model_service": "last_penguin_catalog.nimbus.nimbus_coding_agent_gateway",
  "request_id": "bd27243a-5ea9-4658-a406-1c7ba7148f77",
  "result": "PASSED_EXECUTED_UCODE_THROUGH_UNITY_AI_GATEWAY",
  "thread_id": "01a04212-3d6c-72e2-aa7f-f2bbf8173e72",
  "ucode_exit_code": 0
}
